# Test inference

Attach the soil competition data and the NEW soil-cv-utils code dataset version
(the ZIP built from source hash 4b60a00c, not v10). Enable Internet and the
WANDB_API_KEY secret. A GPU makes inference faster.
Set CHECKPOINTS below, then Save & Run All.
CSV files are saved in /kaggle/working/submissions.


In [1]:
from pathlib import Path

# Part 2 (multiscale / tiles, EMD-only 2-3 epoch ensembles), top 5 by OOF EMD.
# ":latest" is the newest version of each fold artifact; the manifest records exact versions.
CHECKPOINTS = [
    {
        "name": "swin-tiles-constant-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/swin-tiles-constant-fold-0:latest",
            "public-team/soil-grain-size/swin-tiles-constant-fold-1:latest",
            "public-team/soil-grain-size/swin-tiles-constant-fold-2:latest",
            "public-team/soil-grain-size/swin-tiles-constant-fold-3:latest"
        ]
    },
    {
        "name": "swin-tiles-cosine-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/swin-tiles-cosine-fold-0:latest",
            "public-team/soil-grain-size/swin-tiles-cosine-fold-1:latest",
            "public-team/soil-grain-size/swin-tiles-cosine-fold-2:latest",
            "public-team/soil-grain-size/swin-tiles-cosine-fold-3:latest"
        ]
    },
    {
        "name": "vit-scratch-p32-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/vit-scratch-p32-fold-0:latest",
            "public-team/soil-grain-size/vit-scratch-p32-fold-1:latest",
            "public-team/soil-grain-size/vit-scratch-p32-fold-2:latest",
            "public-team/soil-grain-size/vit-scratch-p32-fold-3:latest"
        ]
    },
    {
        "name": "swin-full-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/swin-full-fold-0:latest",
            "public-team/soil-grain-size/swin-full-fold-1:latest",
            "public-team/soil-grain-size/swin-full-fold-2:latest",
            "public-team/soil-grain-size/swin-full-fold-3:latest"
        ]
    },
    {
        "name": "swin-tiles-frozen-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/swin-tiles-frozen-fold-0:latest",
            "public-team/soil-grain-size/swin-tiles-frozen-fold-1:latest",
            "public-team/soil-grain-size/swin-tiles-frozen-fold-2:latest",
            "public-team/soil-grain-size/swin-tiles-frozen-fold-3:latest"
        ]
    }
]
DEVICE = "auto"
OUTPUT_DIR = Path("/kaggle/working/submissions")


## Load the code

Use a run ID, a run URL, or an exact W&B artifact for one model.
Use a named group of four checkpoints for a four-fold ensemble.
Use None for an empty slot.
The package checks model settings, source compatibility and fold IDs.

In [2]:
import os
import sys

import torch
import wandb
from kaggle_secrets import UserSecretsClient

code_root = Path(os.environ.get(
    "SOILCV_INFERENCE_CODE_ROOT",
    "/kaggle/input/datasets/denysmykhailov/soil-cv-utils",
))
if not (code_root / "soilcv/inference/runner.py").is_file():
    matches = list(Path("/kaggle/input").rglob("soilcv/inference/runner.py"))
    if len(matches) != 1:
        raise RuntimeError("Attach the soil-cv-utils code dataset.")
    code_root = matches[0].parents[2]
sys.path.insert(0, str(code_root))
from soilcv.artifacts import package_identity
from soilcv.inference.runner import run_checkpoints


try:
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as error:
    raise RuntimeError("Enable WANDB_API_KEY in Add-ons > Secrets, then Run All.") from error
os.environ["WANDB_API_KEY"] = key
wandb.login(key=key, verify=True)
del key
print("GPU available:", torch.cuda.is_available())
print("Code:", package_identity())


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dmykhailov (dmykhailov-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU available: True
Code: {'version': '0.1.1', 'source_hash': '4b60a00c640d9f745f30841d63291ccc5b5cbd61be6232d2bd803798c2d2aeac', 'folds_hash': '057fe8f93ba35d2ab7a3483063c95ca0079c4bcbb9978671c16ffc4119d9c1b2'}


## Predict the test set

Weights come from W&B. Each model uses its saved input settings.
A four-fold group averages the curves for each test sample.
Rows follow sample_submission.csv. IDs and valid curves are checked.
A group is skipped if a checkpoint is not ready.

In [3]:
results = run_checkpoints(CHECKPOINTS, OUTPUT_DIR, device=DEVICE)
print("Saved CSV files:", len(results))

[1] Resolve 4 checkpoint(s)


wandb: Downloading large artifact 'swin-tiles-constant-fold-0:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.4 (33.6MB/s)
wandb: Downloading large artifact 'swin-tiles-constant-fold-1:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:10.3 (30.6MB/s)
wandb: Downloading large artifact 'swin-tiles-constant-fold-2:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:10.3 (30.8MB/s)
wandb: Downloading large artifact 'swin-tiles-constant-fold-3:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:10.8 (29.3MB/s)


Saved: submission_01_swin-tiles-constant-4fold.csv (10, 12)
[2] Resolve 4 checkpoint(s)


wandb: Downloading large artifact 'swin-tiles-cosine-fold-0:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.3 (34.0MB/s)
wandb: Downloading large artifact 'swin-tiles-cosine-fold-1:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.1 (34.6MB/s)
wandb: Downloading large artifact 'swin-tiles-cosine-fold-2:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.9 (31.8MB/s)
wandb: Downloading large artifact 'swin-tiles-cosine-fold-3:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:08.9 (35.4MB/s)


Saved: submission_02_swin-tiles-cosine-4fold.csv (10, 12)
[3] Resolve 4 checkpoint(s)


wandb:   8 of 8 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   8 of 8 files downloaded.  
wandb:   8 of 8 files downloaded.  


Saved: submission_03_vit-scratch-p32-4fold.csv (10, 12)
[4] Resolve 4 checkpoint(s)


wandb: Downloading large artifact 'swin-full-fold-0:latest', 316.00MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.2 (34.4MB/s)
wandb: Downloading large artifact 'swin-full-fold-1:latest', 316.00MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.7 (32.5MB/s)
wandb: Downloading large artifact 'swin-full-fold-2:latest', 316.00MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.8 (32.1MB/s)
wandb: Downloading large artifact 'swin-full-fold-3:latest', 316.00MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:10.4 (30.4MB/s)


Saved: submission_04_swin-full-4fold.csv (10, 12)
[5] Resolve 4 checkpoint(s)


wandb: Downloading large artifact 'swin-tiles-frozen-fold-0:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.7 (32.7MB/s)
wandb: Downloading large artifact 'swin-tiles-frozen-fold-1:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.9 (31.8MB/s)
wandb: Downloading large artifact 'swin-tiles-frozen-fold-2:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:10.4 (30.4MB/s)
wandb: Downloading large artifact 'swin-tiles-frozen-fold-3:latest', 315.90MB. 8 files...
wandb:   8 of 8 files downloaded.  
Done. 00:00:09.9 (31.9MB/s)


Saved: submission_05_swin-tiles-frozen-4fold.csv (10, 12)
Saved CSV files: 5


## Ready submissions

One CSV belongs to one model or one complete four-fold ensemble.
inference_manifest.json records the model versions and source runs.
Files appear in Kaggle Outputs after Save & Run All ends.
The notebook does not train models or submit files to the competition.

In [4]:
import pandas as pd
from IPython.display import FileLink, display

assert results, "No ready predictions yet."
for result in results:
    path = Path(result["output"])
    frame = pd.read_csv(path)
    print(path.name, "| shape:", frame.shape)
    display(FileLink(str(path)))
    display(frame.head(2))
display(pd.DataFrame(results)[["slot", "name", "ensemble", "device", "output"]])
print("Outputs:", OUTPUT_DIR)


submission_01_swin-tiles-constant-4fold.csv | shape: (10, 12)


/kaggle/working/submissions/submission_01_swin-tiles-constant-4fold.csv

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,2.867580,6.414014,17.241838,51.781725,87.850569,96.060467,97.095570,98.335729,99.476530,99.965544,100.0
1,HPC_Audorfring,2.569105,5.545133,13.779987,34.261810,60.058293,74.330935,80.067462,88.354805,96.361641,99.865088,100.0


submission_02_swin-tiles-cosine-4fold.csv | shape: (10, 12)


/kaggle/working/submissions/submission_02_swin-tiles-cosine-4fold.csv

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,2.718363,6.156704,16.278561,49.625614,85.951197,95.399396,96.703063,98.201308,99.439297,99.964367,100.0
1,HPC_Audorfring,2.591085,5.663982,13.529601,30.550292,50.399891,65.546642,73.537071,84.472099,95.348823,99.892750,100.0


... (7 outputs left out)

/kaggle/working/submissions/submission_05_swin-tiles-frozen-4fold.csv

,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,1.254052,2.301374,6.629490,24.864061,61.289328,87.268061,89.974279,96.382756,98.920169,99.979234,100.0
1,HPC_Audorfring,1.174803,2.258203,7.301702,23.025054,40.465223,60.103917,70.366028,89.037646,96.654276,99.962989,100.0


,slot,name,ensemble,device,output
0,1,swin-tiles-constant-4fold,True,cuda:0,/kaggle/working/submissions/submission_01_swin...
1,2,swin-tiles-cosine-4fold,True,cuda:0,/kaggle/working/submissions/submission_02_swin...
2,3,vit-scratch-p32-4fold,True,cuda:0,/kaggle/working/submissions/submission_03_vit-...
3,4,swin-full-4fold,True,cuda:0,/kaggle/working/submissions/submission_04_swin...
4,5,swin-tiles-frozen-4fold,True,cuda:0,/kaggle/working/submissions/submission_05_swin...


Outputs: /kaggle/working/submissions
